In [1]:
import pandas as pd
import os
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

In [2]:
pd.set_option('display.max_columns', None)

#### <span style="color:#FF6347;">**READ**</span> mo_cancer_dicom

In [37]:
file_path = os.path.join("../../../Dataset/MO-DBT-data-curation", "Cancer", "MO cancer", 'mo_cancer_dicom' + ".xlsx")
df1 = pd.read_excel(file_path)

In [38]:
file_path = os.path.join("../../../Dataset/MO-DBT-data-curation", "Cancer", "Eligible MO cancer from candidate", 'eligible_mo_cancer_dicom' + ".xlsx")
df2 = pd.read_excel(file_path)

In [53]:
df = pd.concat([df1, df2]).reset_index(drop=True)

#### <span style="color:cyan;">**Define target root path and cohort type**</span> (Cancer OR Normal)

In [35]:
root_path = 'J:/Testing/jlee/MO-DBT-Eligible'
cohort = 'Cancer'

## **Step 1**: Construct target folder path

In [54]:
df.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,PatientBirthDate,StudyDate,Study,SIDE,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,time_to_INDEX_months,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,Ki-67_Percent,Ki-67_Grade,subtype,FolderPath
0,4330421371,45,65330676,1973-07-01,2018-12-03,SCREEN,L,C VIEW,CC,NaN,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,L CC C-View,71300000,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-12-03,NaN,NaN,NaN,INDEX-1,31.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330421371,45,65330676,1973-07-01,2018-12-03,SCREEN,L,C VIEW,MLO,NaN,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,L MLO C-View,71300000,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-12-03,NaN,NaN,NaN,INDEX-1,31.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330421371,45,65330676,1973-07-01,2018-12-03,SCREEN,L,C VIEW,CC,NaN,SCREENING MAMMOGRAM WITH TOMO,L CC C-View,71300000,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-12-03,NaN,NaN,NaN,INDEX-1,31.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...
3,4330421371,45,65330676,1973-07-01,2018-12-03,SCREEN,L,C VIEW,MLO,NaN,SCREENING MAMMOGRAM WITH TOMO,L MLO C-View,71300000,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-12-03,NaN,NaN,NaN,INDEX-1,31.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...
4,4330421371,45,65330676,1973-07-01,2018-12-03,SCREEN,L,DBT,CC,NaN,SCREENING MAMMOGRAM WITH TOMO,L CC Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-12-03,NaN,NaN,NaN,INDEX-1,31.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...


In [55]:
DENSITY_SHORT = {
    'Almost entirely fat (<25%)': 'A',
    'Scattered fibroglandular (25% - 50%)': 'B',
    'Heterogeneously dense (51% - 75%)': 'C',
    'Extremely dense (>75%)': 'D'
}

In [56]:
df['Density'] = df['COMPOSITION_NAME'].map(DENSITY_SHORT)

In [57]:
SEP = os.sep

sec_pid = df['PATIENT_STUDY_ID'].astype(str, errors='ignore')
sec_index = df['Index']
sec_density = df['Density']
sec_side = df['SIDE']
sec_view = df['View']
sec_series = df['Series']

# pid / Index (INDEX, INDEX-1, INDEX-2) / birads density / side view / image type (DBT, IN2D, C VIEW)
df['TargetFolderPath'] = root_path + SEP + cohort + SEP + sec_pid + SEP + sec_index + SEP + sec_density + SEP + sec_side + sec_view + SEP + sec_series

In [59]:
df['FolderPath'][3], df['TargetFolderPath'][3]

('J:\\Testing\\jlee\\MO_DBT\\1.21TB_Tomos_Round_12\\Patient_4330421371\\20181203\\Study_SCREENING_MAMMOGRAM_WITH_TOMO_65330676\\Series_71300000_L_MLO_C-View',
 'J:/Testing/jlee/MO-DBT-Eligible\\Cancer\\4330421371\\INDEX-1\\C\\LMLO\\C VIEW')

In [60]:
df_filter = df[~df['TargetFolderPath'].isna()]

In [61]:
df_filter["PATIENT_STUDY_ID"].nunique()

264

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_dicom**

In [62]:
output_file = os.path.join(root_path, 'mo_cancer_dicom' + ".xlsx")
df_filter.to_excel(output_file, index=False)

## **Step 2**: Copy files

In [63]:
df_input = df_filter.copy()

In [ ]:
for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc='move files'):
    source_dir = Path(row['FolderPath'])
    target_dir = Path(row['TargetFolderPath'])
            
    if not source_dir.is_dir():
        print(f"WARNING: Source directory not found: {source_dir}")
        continue

    if not target_dir.is_dir():
        try:
            target_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  CRITICAL ERROR: Could not create target directory {target_dir.name}: {e}")
            continue
        
    # Iterate over ALL files within the source directory
    for item in source_dir.iterdir():
        source_item = item # The full path of the file/folder to move
        target_item = target_dir / item.name # The full path where it will land

        # print(source_item, target_item)

        try:
            # Check if it's a directory or a file to use the correct copy function
            if source_item.is_dir():
                # copytree copies the entire directory and its contents
                shutil.copytree(source_item, target_item, dirs_exist_ok=True)
            else:
                # copy2 copies the file and preserves metadata (like timestamps)
                shutil.copy2(source_item, target_item)
                
            # print(f"  COPIED: {item.name}")
            
        except Exception as e:
            print(f"  ERROR copying {item.name}: {e}")



        # try:
        #     # This handles both files and directories
        #     shutil.move(source_item, target_item)
        #     # print(f"  MOVED: {item.name}")
            
        # except Exception as e:
        #     print(f"  ERROR moving {item.name}: {e}")

move files:   0%|          | 0/5956 [00:00<?, ?it/s]

move files:   2%|▏         | 119/5956 [05:48<4:45:00,  2.93s/it] 